<a href="https://colab.research.google.com/github/Aymane-Aziz/toxic-comment-classifier/blob/main/03_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, roc_auc_score, hamming_loss
import warnings
warnings.filterwarnings('ignore')

LABEL_COLS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
BASE_PATH  = '/content/drive/MyDrive/toxic-comment-classifier/data/'

train_df = pd.read_csv(BASE_PATH + 'train_processed.csv')
val_df   = pd.read_csv(BASE_PATH + 'val_processed.csv')

print(f"Train: {train_df.shape}, Val: {val_df.shape}")

Mounted at /content/drive
Train: (135635, 12), Val: (23936, 12)


In [4]:
# Drop or fill NaN values in cleaned_text
train_df['cleaned_text'] = train_df['cleaned_text'].fillna('')
val_df['cleaned_text']   = val_df['cleaned_text'].fillna('')

print(f"NaNs in train: {train_df['cleaned_text'].isna().sum()}")
print(f"NaNs in val:   {val_df['cleaned_text'].isna().sum()}")

NaNs in train: 0
NaNs in val:   0


In [5]:
# TF-IDF converts raw text into numerical features
# sublinear_tf dampens the effect of very frequent words
# ngram_range=(1,2) captures single words AND two-word phrases

vectorizer = TfidfVectorizer(
    max_features=50000,
    sublinear_tf=True,
    ngram_range=(1, 2),
    min_df=3,
    strip_accents='unicode',
    analyzer='word'
)

X_train = vectorizer.fit_transform(train_df['cleaned_text'])
X_val   = vectorizer.transform(val_df['cleaned_text'])

y_train = train_df[LABEL_COLS].values
y_val   = val_df[LABEL_COLS].values

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")

X_train shape: (135635, 50000)
X_val shape:   (23936, 50000)


In [6]:
# OneVsRestClassifier trains one classifier per label
# This is the standard approach for multi-label classification

model = OneVsRestClassifier(
    LogisticRegression(
        C=5,
        max_iter=1000,
        class_weight='balanced',  # handles imbalance
        solver='lbfgs'
    )
)

print("Training baseline model...")
model.fit(X_train, y_train)
print("Done!")

Training baseline model...
Done!


In [7]:
y_pred  = model.predict(X_val)
y_proba = model.predict_proba(X_val)

# ── Per-label metrics ────────────────────────────────────
print("=" * 55)
print(f"{'Label':<18} {'F1':>8} {'ROC-AUC':>10}")
print("=" * 55)

for i, label in enumerate(LABEL_COLS):
    f1      = f1_score(y_val[:, i], y_pred[:, i])
    roc_auc = roc_auc_score(y_val[:, i], y_proba[:, i])
    print(f"{label:<18} {f1:>8.4f} {roc_auc:>10.4f}")

# ── Overall metrics ──────────────────────────────────────
print("=" * 55)
print(f"\nOverall F1     (macro): {f1_score(y_val, y_pred, average='macro'):.4f}")
print(f"Overall ROC-AUC(macro): {roc_auc_score(y_val, y_proba, average='macro'):.4f}")
print(f"Hamming Loss:           {hamming_loss(y_val, y_pred):.4f}")

Label                    F1    ROC-AUC
toxic                0.7679     0.9742
severe_toxic         0.4380     0.9831
obscene              0.7846     0.9846
threat               0.4179     0.9831
insult               0.7047     0.9792
identity_hate        0.4177     0.9750

Overall F1     (macro): 0.5885
Overall ROC-AUC(macro): 0.9799
Hamming Loss:           0.0250


In [8]:
# Store results so we can compare with RoBERTa later
baseline_results = {
    'model'    : 'TF-IDF + Logistic Regression',
    'f1_macro' : f1_score(y_val, y_pred, average='macro'),
    'roc_auc'  : roc_auc_score(y_val, y_proba, average='macro'),
    'hamming'  : hamming_loss(y_val, y_pred)
}

print("Baseline results saved:")
for k, v in baseline_results.items():
    print(f"  {k}: {v}")

Baseline results saved:
  model: TF-IDF + Logistic Regression
  f1_macro: 0.588497341968247
  roc_auc: 0.979874811405988
  hamming: 0.025025066844919786
